In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "gpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import jax
print(jax.devices())
print("default backend:", jax.default_backend())


[CudaDevice(id=0)]
default backend: gpu


In [3]:
from alphagenome.data import genome
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models import variant_scorers
from alphagenome.models import dna_client

import matplotlib.pyplot as plt
import functools
import os
from typing import Callable

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn import metrics
from sklearn.metrics import average_precision_score, roc_auc_score


I0000 00:00:1777463906.802983 4047418 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777463910.366842 4047418 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# @title Eval configs.
url = "https://storage.googleapis.com/alphagenome/evals/clinvar_noncoding_predictions.feather"
df = pd.read_feather(url)

df.head()


,variant_id,prediction,target,variant_scorer,output_type,metric_calculator,metric_name
0,chr1_1043197_G_C_hg38,0.016315,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
1,chr1_2520467_C_T_hg38,0.006831,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
2,chr1_9943525_C_T_hg38,0.040409,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
3,chr1_11790916_C_T_hg38,0.811025,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation
4,chr1_11998739_T_G_hg38,0.343018,1.0,"ensemble_scorer;sj,ssu,ss",RNA_SEQ;SPLICE_SITE_POSITIONS;SPLICE_SITE_USAG...,SimpleAggregatorAUPRCCalculator,auprc_max_abs_track_aggregation


In [5]:
def parse_variant(variant_id):
    parts = variant_id.split("_")
    if len(parts) >= 4:
        chrom, pos, ref, alt = parts[:4]
        return ref, alt
    return None, None

def classify_variant(variant_id):
    ref, alt = parse_variant(variant_id)
    
    if ref is None:
        return "unknown"
    
    if len(ref) == len(alt) == 1:
        return "snp"
    elif len(alt) > len(ref):
        return "insertion"
    elif len(ref) > len(alt):
        return "deletion"
    else:
        return "other"
    
df["variant_type"] = df["variant_id"].apply(classify_variant)

insertions = df[df["variant_type"] == "insertion"].drop_duplicates("variant_id").head(100)
deletions = df[df["variant_type"] == "deletion"].drop_duplicates("variant_id").head(100)

variant_insertions = insertions["variant_id"].tolist()
variant_deletions = deletions["variant_id"].tolist()

In [6]:
variant_scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ'] 

# Flags to improve determinism.
os.environ['XLA_FLAGS'] = ' '.join([
    '--xla_gpu_deterministic_ops',
    '--xla_gpu_enable_scatter_determinism_expander=True',
    '--xla_gpu_enable_triton_gemm=False',
])
# Increase GPU and CPU memory to reduce out of memory errors.
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'

model = dna_model.create_from_huggingface('all_folds')

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/opt/modules/i12g/anaconda/envs/IDPproject_indel_26/lib/python3.11/site-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


In [7]:
def score_variant(variant_id: str) -> pd.DataFrame: 

    parts = variant_id.split("_")

    if len(parts) == 5:
        chrom, pos, ref, alt, ext = parts
    elif len(parts) == 4:
        chrom, pos, ref, alt = parts
        ext = None
    else:
        raise ValueError(f"Unexpected variant_id format: {variant_id}")

    # build variant object
    variant = genome.Variant(
        chromosome=chrom,
        position=int(pos),
        reference_bases=ref,
        alternate_bases=alt,
    )

        
    # Create a 1MB interval centered on the variant.
    interval = variant.reference_interval.resize(2**20)

    variant_scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ'] 
    variant_scores = model.score_variant(
        interval=interval, 
        variant=variant, 
        variant_scorers=[variant_scorer],
        organism=dna_client.Organism.HOMO_SAPIENS,
    )
    variant_scores = variant_scorers.tidy_scores(variant_scores)
    
    return variant_scores

In [8]:

unique_variants = insertions["variant_id"]

variant_scores_dic = {}

from tqdm import tqdm
for variant in tqdm(unique_variants):
    variant_scores_dic[variant] = score_variant(variant)



  0%|          | 0/100 [00:00<?, ?it/s]E0429 14:00:19.958059 4047952 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.452 = bf16[16384,192]{1,0} fusion(bitcast.1680, bitcast.2090, v_layer____w__.9), kind=kCustom, calls=gemm_fusion_dot.452_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0429 14:00:19.958190 4047952 xtile_compiler.cc:401] Computation: gemm_fusion_dot.452_computation.clone {
  parameter_0 = bf16[1536,8192]{1,0} parameter(0)
  parameter_1 = bf16[1536,8192]{1,0} parameter(1)
  concatenate.460 = bf16[1536,16384]{1,0} concatenate(parameter_0, parameter_1), dimensions={1}
  bitcast.7774 = bf16[16384,1536]{0,1} b

In [9]:
# collect output 

insertions_results_local = []

for variant in variant_scores_dic:
    df = variant_scores_dic[variant]
    if df is None or df.empty:
        continue
    insertions_results_local.append(df)

# convert list of rows → DataFrame
insertions_results_local = pd.concat(insertions_results_local, ignore_index=True)
    


In [16]:
# filter out those with no gtx data

insertions_results_local = insertions_results_local[insertions_results_local["gtex_tissue"] != ""]

In [17]:
cols = ["variant_id", "scored_interval", "ontology_curie", "gene_id", "gtex_tissue", "raw_score"]

df_save_insertions = insertions_results_local[cols].copy()

df_save_insertions["variant_id"] = df_save_insertions["variant_id"].astype(str)
df_save_insertions["scored_interval"] = df_save_insertions["scored_interval"].astype(str)

df_save_insertions.to_feather("results/indel_compare/local_insertions.feather")

In [18]:
df_save_insertions = pd.read_feather("results/indel_compare/local_insertions.feather")

In [19]:
df_save_insertions

,variant_id,scored_interval,ontology_curie,gene_id,gtex_tissue,raw_score
296,chr2:60461136:G>GCCA,chr2:59936848-60985424:.,EFO:0000572,ENSG00000271955,Cells_EBV-transformed_lymphocytes,0.001047
299,chr2:60461136:G>GCCA,chr2:59936848-60985424:.,EFO:0002009,ENSG00000271955,Cells_Cultured_fibroblasts,0.000024
317,chr2:60461136:G>GCCA,chr2:59936848-60985424:.,UBERON:0000007,ENSG00000271955,Pituitary,-0.000354
319,chr2:60461136:G>GCCA,chr2:59936848-60985424:.,UBERON:0000458,ENSG00000271955,Cervix_Endocervix,-0.000175
320,chr2:60461136:G>GCCA,chr2:59936848-60985424:.,UBERON:0000473,ENSG00000271955,Testis,0.005777
...,...,...,...,...,...,...
1717839,chr1:68446872:G>GGGA,chr1:67922584-68971160:.,UBERON:0010414,ENSG00000285407,Adipose_Visceral_Omentum,0.028894
1717840,chr1:68446872:G>GGGA,chr1:67922584-68971160:.,UBERON:0011907,ENSG00000285407,Muscle_Skeletal,0.053307
1717841,chr1:68446872:G>GGGA,chr1:67922584-68971160:.,UBERON:0012249,ENSG00000285407,Cervix_Ectocervix,0.062244
1717842,chr1:68446872:G>GGGA,chr1:67922584-68971160:.,UBERON:0013756,ENSG00000285407,Whole_Blood,0.006587


## Deletion local

In [20]:

unique_variants = deletions["variant_id"]

variant_scores_dic = {}

from tqdm import tqdm
for variant in tqdm(unique_variants):
    variant_scores_dic[variant] = score_variant(variant)



100%|██████████| 100/100 [05:48<00:00,  3.49s/it]


In [21]:
# only take one gene for each variant

deletions_results_local = []


for variant in variant_scores_dic:
    df = variant_scores_dic[variant]
    if df is None or df.empty:
        continue
    deletions_results_local.append(df)

# convert list of rows → DataFrame
deletions_results_local = pd.concat(deletions_results_local, ignore_index=True)
    

In [22]:
# filter out those with no gtx data

deletions_results_local = deletions_results_local[deletions_results_local["gtex_tissue"] != ""]

In [23]:
cols = ["variant_id", "scored_interval", "ontology_curie", "gene_id", "gtex_tissue", "raw_score"]

df_save_deletions = deletions_results_local[cols].copy()

df_save_deletions["variant_id"] = df_save_deletions["variant_id"].astype(str)
df_save_deletions["scored_interval"] = df_save_deletions["scored_interval"].astype(str)

df_save_deletions.to_feather("results/indel_compare/local_deletions.feather")

In [24]:
df_save_deletions

,variant_id,scored_interval,ontology_curie,gene_id,gtex_tissue,raw_score
296,chr1:37708311:TTTC>T,chr1:37184024-38232600:.,EFO:0000572,ENSG00000223944,Cells_EBV-transformed_lymphocytes,0.000398
299,chr1:37708311:TTTC>T,chr1:37184024-38232600:.,EFO:0002009,ENSG00000223944,Cells_Cultured_fibroblasts,-0.000017
317,chr1:37708311:TTTC>T,chr1:37184024-38232600:.,UBERON:0000007,ENSG00000223944,Pituitary,0.002805
319,chr1:37708311:TTTC>T,chr1:37184024-38232600:.,UBERON:0000458,ENSG00000223944,Cervix_Endocervix,0.000667
320,chr1:37708311:TTTC>T,chr1:37184024-38232600:.,UBERON:0000473,ENSG00000223944,Testis,0.000226
...,...,...,...,...,...,...
1224027,chr11:45910267:TAGA>T,chr11:45385980-46434556:.,UBERON:0010414,ENSG00000244313,Adipose_Visceral_Omentum,-0.034138
1224028,chr11:45910267:TAGA>T,chr11:45385980-46434556:.,UBERON:0011907,ENSG00000244313,Muscle_Skeletal,-0.026610
1224029,chr11:45910267:TAGA>T,chr11:45385980-46434556:.,UBERON:0012249,ENSG00000244313,Cervix_Ectocervix,-0.035342
1224030,chr11:45910267:TAGA>T,chr11:45385980-46434556:.,UBERON:0013756,ENSG00000244313,Whole_Blood,-0.031347
